In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Arabert Model

In [ ]:
!pip install transformers[torch] datasets arabert
!pip install nlpaug
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from arabert.preprocess import ArabertPreprocessor
from datasets import Dataset

# Check GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using Device: {device}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.0/185.0 kB 7.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.4/126.4 kB 15.6 MB/s eta 0:00:00
  Created wheel for emoji: filename=emoji-1.4.2-py3-none-any.whl size=186456 sha256=c9fe4ead20522f6471b363889fe4143ba97b3159ae3e27ef0250df9676bd5085
  Stored in directory: /root/.cache/pip/wheels/bb/f1/26/f9002669ef6ad80a3c9f1b22880b35d9b4c6650011acee0523
Successfully built emoji
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.5/410.5 kB 10.2 MB/s eta 0:00:00
Using Device: cuda


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os

DRIVE_DIR = "/content/drive/MyDrive/hajj_real_data"
os.makedirs(DRIVE_DIR, exist_ok=True)

IMAGES_DIR = os.path.join(DRIVE_DIR, "images")
os.makedirs(IMAGES_DIR, exist_ok=True)

CSV_PATH = os.path.join(DRIVE_DIR, "real_data.csv")

print(f"[OK] drive connected")
print(f"[OK] csv path: {CSV_PATH}")
print(f"[OK] images path: {IMAGES_DIR}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[OK] drive connected
[OK] csv path: /content/drive/MyDrive/hajj_real_data/real_data.csv
[OK] images path: /content/drive/MyDrive/hajj_real_data/images


In [ ]:
import re
import os
import pandas as pd
import torch
import nlpaug.augmenter.word as naw

from datasets import Dataset, DatasetDict
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from transformers import (
    TrainingArguments,
    Trainer,
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding
)

# =========================
# 1. LOAD DATA
# =========================
old_fake = pd.read_csv("/content/drive/MyDrive/hajj_processing_workspace/fake_data_ready.csv")
old_real = pd.read_csv("/content/drive/MyDrive/hajj_processing_workspace/real_data_ready.csv")

ckpt_path = "/content/drive/MyDrive/hajj_processing_workspace/advanced_fake_generation/checkpoints"
new_fake_files = sorted([f for f in os.listdir(ckpt_path) if f.endswith('.csv')])

new_fake_df = pd.concat(
    [pd.read_csv(os.path.join(ckpt_path, f)) for f in new_fake_files],
    ignore_index=True
).iloc[:900]

all_fakes = pd.concat([old_fake, new_fake_df], ignore_index=True)
all_data = pd.concat([all_fakes, old_real], ignore_index=True)

print(f"Total Data: {len(all_data)}")

# =========================
# 2. LABEL ENCODING
# =========================
le = LabelEncoder()
all_data['Label_Encoded'] = le.fit_transform(all_data['Label'])

# =========================
# 3. CLEAN TEXT (light only)
# =========================
def clean_text(text):
    text = str(text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

all_data['Bert_Processed'] = (
    all_data['Text Content'].fillna('') + " " +
    all_data['Image Description'].fillna('')
).apply(clean_text)

# =========================
# 4. SPLIT FIRST (VERY IMPORTANT)
# =========================
train_df, test_df = train_test_split(
    all_data,
    test_size=0.2,
    stratify=all_data['Label'],
    random_state=42
)

train_texts = set(train_df['Bert_Processed'])
test_texts = set(test_df['Bert_Processed'])
overlap = train_texts.intersection(test_texts)

print(f"Overlap check before cleanup: Found {len(overlap)} overlapping sentences between train and test.")

if len(overlap) > 0:
    print("Removing overlapping samples from train set...")
    train_df = train_df[~train_df['Bert_Processed'].isin(test_texts)]

    new_train_texts = set(train_df['Bert_Processed'])
    new_overlap = new_train_texts.intersection(test_texts)
    print(f"Overlap check after cleanup: {len(new_overlap)} overlapping sentences.")

# =========================
# 5. SAFE AUGMENTATION (TRAIN ONLY REAL CLASS)
# =========================
aug = naw.SynonymAug(aug_src='wordnet')

def augment_text(x):
    try:
        return aug.augment(str(x))
    except:
        return x

train_real = train_df[train_df['Label'] == 'Legitimate'].copy()

train_real_aug = train_real.copy()
train_real_aug['Bert_Processed'] = train_real_aug['Bert_Processed'].apply(augment_text)

train_fake = train_df[train_df['Label'] == 'Suspicious']

train_df_final = pd.concat([train_fake, train_real, train_real_aug], ignore_index=True)

# =========================
# 6. MODEL
# =========================
model_name = "aubmindlab/bert-base-arabertv2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# =========================
# 7. DATASET
# =========================
train_dataset = Dataset.from_pandas(train_df_final[['Bert_Processed', 'Label_Encoded']])
test_dataset = Dataset.from_pandas(test_df[['Bert_Processed', 'Label_Encoded']])

train_dataset = train_dataset.rename_column("Bert_Processed", "text").rename_column("Label_Encoded", "label")
test_dataset = test_dataset.rename_column("Bert_Processed", "text").rename_column("Label_Encoded", "label")

if "__index_level_0__" in train_dataset.column_names:
    train_dataset = train_dataset.remove_columns(["__index_level_0__"])
if "__index_level_0__" in test_dataset.column_names:
    test_dataset = test_dataset.remove_columns(["__index_level_0__"])

dataset = DatasetDict({
    'train': train_dataset,
    'test': test_dataset
})

print("\n=============================")
print("Final Dataset Sizes:")
print("=============================")
print(f"Train size (after augmentation): {len(dataset['train'])}")
print(f"Test size (clean): {len(dataset['test'])}")
print(f"Total: {len(dataset['train']) + len(dataset['test'])}")

# =========================
# 8. TOKENIZATION
# =========================
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding=True,
        max_length=128
    )

tokenized = dataset.map(tokenize_function, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


Total Data: 2843
Overlap check before cleanup: Found 1 overlapping sentences between train and test.
Removing overlapping samples from train set...
Overlap check after cleanup: 0 overlapping sentences.


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is al

config.json:   0%|          | 0.00/384 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/611 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: aubmindlab/bert-base-arabertv2
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.embeddings.position_ids               | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider


Final Dataset Sizes:
Train size (after augmentation): 3069
Test size (clean): 569
Total: 3638


Map:   0%|          | 0/3069 [00:00<?, ? examples/s]

Map:   0%|          | 0/569 [00:00<?, ? examples/s]

In [ ]:
# =========================
# 9. TRAINING
# =========================
training_args = TrainingArguments(
    output_dir="./arabert_model_aug",
    num_train_epochs=2,
    learning_rate=1e-5,
    per_device_train_batch_size=16,

    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,

    weight_decay=0.05,
    warmup_ratio=0.1,

    fp16=True,
    logging_steps=50,
    save_total_limit=2,

    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    data_collator=data_collator
)

print(" Training AraBERT with SAFE augmentation...")
trainer.train()

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


 Training AraBERT with SAFE augmentation...


Epoch,Training Loss,Validation Loss
1,0.021128,0.027572
2,0.001901,0.030774


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=384, training_loss=0.08321508945664391, metrics={'train_runtime': 78.5068, 'train_samples_per_second': 78.184, 'train_steps_per_second': 4.891, 'total_flos': 403743914449920.0, 'train_loss': 0.08321508945664391, 'epoch': 2.0})

In [ ]:
import numpy as np
from sklearn.metrics import classification_report

print("Evaluating model...")
predictions = trainer.predict(tokenized["test"])

preds = np.argmax(predictions.predictions, axis=-1)

true_labels = predictions.label_ids

print("\n=========================")
print("  CLASSIFICATION REPORT")
print("=========================\n")
print(classification_report(true_labels, preds))


Evaluating model...



  CLASSIFICATION REPORT

              precision    recall  f1-score   support

           0       1.00      0.98      0.99       200
           1       0.99      1.00      0.99       369

    accuracy                           0.99       569
   macro avg       0.99      0.99      0.99       569
weighted avg       0.99      0.99      0.99       569



In [ ]:
import torch
import torch.nn.functional as F

model.eval()

def predict_ad(text_content, image_description):
    full_context = (text_content or "") + " " + (image_description or "")

    processed_text = full_context.strip()

    inputs = tokenizer(
        processed_text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits
    probabilities = F.softmax(logits, dim=1)

    predicted_class_id = torch.argmax(probabilities, dim=1).item()
    confidence_score = probabilities[0, predicted_class_id].item() * 100

    predicted_label = le.inverse_transform([predicted_class_id])[0]

    return predicted_label, confidence_score


# =========================
# TEST EXAMPLE
# =========================
scam_ad_text = """
بشرى سارة لراغبي الحج! 🕋
حج بتأشيرة زيارة مضمونة 100% وبدون قرعة.
السعر مفاجأة والدفع عبر فودافون كاش أو إنستاباي لسرعة الحجز.
الأماكن محدودة جداً، تواصل معنا الآن عبر الواتساب.
"""

scam_image_desc = "A professional looking poster with a red official stamp, the Kaaba in the background."

label_v2, score_v2 = predict_ad(scam_ad_text, scam_image_desc)

print("=" * 40)
print("MODEL RESULT:")
print(f"PREDICTION: {label_v2}")
print(f"CONFIDENCE: {score_v2:.2f}%")
print("=" * 40)

MODEL RESULT:
PREDICTION: Suspicious
CONFIDENCE: 99.70%


In [ ]:
# --- تجربة العينة الحقيقية (تروبيك للسياحة) على الموديل المحدث ---
legit_ad_text = """حج بكل هدوء وتركيز في العبادة مع تروبيك للسياحة 🕋✨
معنا كل وسائل الانتقال مجهزة خصيصاً لخدمة وراحة حجاجنا
سواء كان قطار الحرمين السريع ،أو فخامة سيارات الـ GMC، أو راحة باصاتنا الحديثة.. طريقك للمشاعر مليئ بالراحة 🧡
راحتك في الطريق، قبل الوصول هدفنا 💙
تروبيك للسياحة .. رحلة حج وعمرة لقلب مطمئن
#تروبيك_للسياحة
#بِثِقَةٍ_نَصِلُ_وبِأَثَرٍ_نَبْقَىٰ
#رحلة_حج_وعمرة_لقلب_مطمئن
#حج_1447هـ"""

legit_image_desc = "اتوبيس سياح امام جبل كبير"

# تشغيل التنبؤ بالموديل V2
label_legit, score_legit = predict_ad(legit_ad_text, legit_image_desc)

print("-" * 40)
print(f"TESTING LEGITIMATE AD (Model V2)...")
print(f"RESULT: {label_legit}")
print(f"CONFIDENCE: {score_legit:.2f}%")
print("-" * 40)

----------------------------------------
TESTING LEGITIMATE AD (Model V2)...
RESULT: Legitimate
CONFIDENCE: 99.81%
----------------------------------------


In [ ]:
import os

BASE_PATH = "/content/drive/MyDrive/final_models_V2"
os.makedirs(BASE_PATH, exist_ok=True)

print("Folder ready:", BASE_PATH)

Folder ready: /content/drive/MyDrive/final_models_V2


In [ ]:
bert_path = os.path.join(BASE_PATH, "arabert_model")

trainer.save_model(bert_path)
tokenizer.save_pretrained(bert_path)

import joblib
joblib.dump(le, os.path.join(bert_path, "label_encoder.pkl"))

print(" AraBERT saved in:", bert_path)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

 AraBERT saved in: /content/drive/MyDrive/final_models_V2/arabert_model


In [1]:
joblib.dump(le, f"{bert_path}/label_encoder.pkl")

NameError: name 'joblib' is not defined